# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 07 — Hyperparameter Tuning

---

### Purpose
Systematically optimise the hyperparameters of each trained classifier using
three complementary strategies — GridSearchCV, RandomizedSearchCV, and Optuna —
then compare their outcomes and save the best tuned models.

### Objectives
1. Load the best-performing feature set identified in Notebooks 03–06
2. Apply **GridSearchCV** — exhaustive grid search
3. Apply **RandomizedSearchCV** — stochastic parameter sampling
4. Apply **Optuna** — Bayesian TPE optimisation
5. Compare performance and tuning efficiency across all strategies
6. Record best hyperparameters per model
7. Save all tuned models

### Workflow
```
Feature Matrix (best set per model)
        │
        ├── GridSearchCV ──────────► best_params (exhaustive)
        ├── RandomizedSearchCV ────► best_params (stochastic)
        └── Optuna ────────────────► best_params (Bayesian)
                │
                ▼
        Comparison Table
                │
                ▼
        Tuned Models Saved → models/tuned/
```

### Expected Outputs
- `models/tuned/logistic_regression_tuned.joblib`
- `models/tuned/linear_svm_tuned.joblib`
- `models/tuned/random_forest_tuned.joblib`
- `models/tuned/xgboost_tuned.joblib`
- `outputs/tuning_results.csv`

### Dependencies
```
scikit-learn >= 1.2.2
optuna >= 3.0
xgboost
joblib
```

### Notebook Outline
1. Imports
2. Configuration
3. Load Best Feature Sets
4. GridSearchCV — Logistic Regression
5. GridSearchCV — Linear SVM
6. RandomizedSearchCV — Random Forest
7. RandomizedSearchCV — XGBoost
8. Optuna — Logistic Regression
9. Optuna — Random Forest
10. Optuna — XGBoost
11. Performance Comparison
12. Best Parameters Summary
13. Save Tuned Models
14. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.tuner import (
    HyperparameterTuner,
    LR_PARAM_GRID, LR_PARAM_DIST,
    SVM_PARAM_GRID,
    RF_PARAM_GRID, RF_PARAM_DIST,
    XGB_PARAM_DIST,
)
from src.feature_engineering.utils import load_feature_matrix, setup_logger
from src.evaluation.evaluator import ModelEvaluator
from src.utils.helpers import (
    set_global_seed, train_test_val_split, load_yaml, make_output_dirs,
    print_section_header, save_json,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED = cfg['random_seed']
TEST_SIZE   = cfg['evaluation']['test_size']
VAL_SIZE    = cfg['evaluation']['val_size']
FEAT_CFG    = cfg['features']
TUNE_CFG    = cfg['tuning']

set_global_seed(RANDOM_SEED)

DIR_MODELS        = PROJECT_ROOT / cfg['output']['models_dir']
DIR_TUNED_MODELS  = DIR_MODELS / 'tuned'
DIR_OUTPUTS       = PROJECT_ROOT / cfg['output']['outputs_dir']
DIR_FIGURES       = PROJECT_ROOT / cfg['output']['figures_dir']
make_output_dirs(DIR_TUNED_MODELS, DIR_OUTPUTS, DIR_FIGURES)

setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])
logger = logging.getLogger(__name__)

# Instantiate tuner
tuner = HyperparameterTuner(cfg=TUNE_CFG)

print(f'Tuning method  : {TUNE_CFG["method"]}')
print(f'CV folds       : {TUNE_CFG["cv_folds"]}')
print(f'Scoring metric : {TUNE_CFG["scoring"]}')
print(f'Optuna trials  : {TUNE_CFG["n_trials"]}')
print(f'Random n_iter  : {TUNE_CFG["n_iter"]}')

---

## 3. Load Best Feature Sets

> **Design decision**: We tune each classifier on its best-performing feature set
> identified in Notebooks 03–06. This is the most computationally efficient
> strategy and avoids cross-contaminating the evaluation.
>
> **Default**: TF-IDF for LR and SVM; embeddings for RF and XGBoost.
> Update `BEST_FEATURE_SETS` based on your Notebook 03–06 results.

In [ ]:
# ── CONFIGURE: Update these based on Notebooks 03–06 results ──────────────────
BEST_FEATURE_SETS = {
    'logistic_regression': 'tfidf',      # typically sparse TF-IDF
    'linear_svm':          'tfidf',      # typically sparse TF-IDF
    'random_forest':       'embedding',  # typically dense embeddings
    'xgboost':             'embedding',  # typically dense embeddings
}

# ── Load TF-IDF (for LR and SVM) ──────────────────────────────────────────────
X_tfidf, y_tfidf = load_feature_matrix(
    PROJECT_ROOT / FEAT_CFG['tfidf']['fingerprint'],
    PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
)
classes = np.load(
    str(PROJECT_ROOT / 'data' / 'features' / 'tfidf' / 'classes_tfidf_fingerprint.npy'),
    allow_pickle=True,
)
X_tr_tf, X_val_tf, X_te_tf, y_tr_tf, y_val_tf, y_te_tf = train_test_val_split(
    X_tfidf, y_tfidf, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)

# ── Load Embeddings (for RF and XGBoost) ──────────────────────────────────────
EMB_DIR = PROJECT_ROOT / 'data' / 'features' / 'embedding'
X_emb   = np.load(str(EMB_DIR / 'emb_fingerprint.npz'))['embeddings']
y_emb   = np.load(str(EMB_DIR / 'labels_emb_fingerprint.npy'))
X_tr_em, X_val_em, X_te_em, y_tr_em, y_val_em, y_te_em = train_test_val_split(
    X_emb, y_emb, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)

# Merge train + val for tuning (CV will handle validation internally)
import scipy.sparse as sp

def vstack(A, B):
    """Stack feature matrices vertically (sparse or dense)."""
    if sp.issparse(A):
        return sp.vstack([A, B], format='csr')
    return np.vstack([A, B])

X_tune_tf = vstack(X_tr_tf, X_val_tf)
y_tune_tf = np.concatenate([y_tr_tf, y_val_tf])

X_tune_em = vstack(X_tr_em, X_val_em)
y_tune_em = np.concatenate([y_tr_em, y_val_em])

print(f'TF-IDF tuning set  : {X_tune_tf.shape}')
print(f'Embedding tuning set: {X_tune_em.shape}')
print(f'Classes             : {list(classes)}')

---

## 4. GridSearchCV — Logistic Regression

> **Strategy**: Exhaustive search over C, solver, and max_iter.
> Grid search is appropriate here because the LR parameter space is small and tractable.

In [ ]:
from sklearn.linear_model import LogisticRegression

print_section_header('GridSearchCV — Logistic Regression (TF-IDF)')

lr_base = LogisticRegression(
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

lr_best_estimator, lr_grid_params, lr_grid_score = tuner.tune_grid(
    estimator=lr_base,
    param_grid=LR_PARAM_GRID,
    X_train=X_tune_tf,
    y_train=y_tune_tf,
    model_name='logistic_regression',
)

print(f'\nBest score  : {lr_grid_score:.4f}')
print(f'Best params : {lr_grid_params}')

---

## 5. GridSearchCV — Linear SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print_section_header('GridSearchCV — Linear SVM (TF-IDF)')

svm_base = CalibratedClassifierCV(
    LinearSVC(
        class_weight='balanced',
        random_state=RANDOM_SEED,
    ),
    cv=3,
    method='sigmoid',
)

svm_best_estimator, svm_grid_params, svm_grid_score = tuner.tune_grid(
    estimator=svm_base,
    param_grid=SVM_PARAM_GRID,
    X_train=X_tune_tf,
    y_train=y_tune_tf,
    model_name='linear_svm',
)

print(f'\nBest score  : {svm_grid_score:.4f}')
print(f'Best params : {svm_grid_params}')

---

## 6. RandomizedSearchCV — Random Forest

> **Strategy**: Random sampling is preferred for RF because its parameter space
> is large (n_estimators × max_depth × min_samples_split × max_features).
> GridSearchCV on this space would take days; random sampling gives a good
> approximation in minutes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier as SKLearnRF

print_section_header(f'RandomizedSearchCV — Random Forest (n_iter={TUNE_CFG["n_iter"]})')

rf_base = SKLearnRF(
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

rf_best_estimator, rf_rand_params, rf_rand_score = tuner.tune_random(
    estimator=rf_base,
    param_distributions=RF_PARAM_DIST,
    X_train=X_tune_em,
    y_train=y_tune_em,
    model_name='random_forest',
)

print(f'\nBest score  : {rf_rand_score:.4f}')
print(f'Best params : {rf_rand_params}')

---

## 7. RandomizedSearchCV — XGBoost

In [ ]:
from xgboost import XGBClassifier

print_section_header(f'RandomizedSearchCV — XGBoost (n_iter={TUNE_CFG["n_iter"]})')

xgb_base = XGBClassifier(
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

xgb_best_estimator, xgb_rand_params, xgb_rand_score = tuner.tune_random(
    estimator=xgb_base,
    param_distributions=XGB_PARAM_DIST,
    X_train=X_tune_em,
    y_train=y_tune_em,
    model_name='xgboost',
)

print(f'\nBest score  : {xgb_rand_score:.4f}')
print(f'Best params : {xgb_rand_params}')

---

## 8. Optuna — Logistic Regression

> **Strategy**: Optuna's TPE sampler learns from previous trials to focus on
> promising regions of parameter space. More efficient than random search for
> a fixed trial budget.

In [ ]:
from sklearn.model_selection import cross_val_score

def lr_optuna_objective(trial):
    """Optuna objective for Logistic Regression hyperparameter optimisation.

    Args:
        trial: optuna.Trial object for parameter suggestion.

    Returns:
        Mean cross-validated macro F1 score.
    """
    C      = trial.suggest_float('C', 0.001, 100.0, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga'])

    model = LogisticRegression(
        C=C,
        solver=solver,
        max_iter=2000,
        class_weight='balanced',
        multi_class='auto',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    scores = cross_val_score(
        model, X_tune_tf, y_tune_tf,
        cv=TUNE_CFG['cv_folds'],
        scoring=TUNE_CFG['scoring'],
        n_jobs=1,   # avoid nested parallelism
    )
    return float(np.mean(scores))

print_section_header(f'Optuna — Logistic Regression (n_trials={TUNE_CFG["n_trials"]})')
lr_optuna_params, lr_optuna_score = tuner.tune_optuna(
    objective_fn=lr_optuna_objective,
    model_name='logistic_regression_optuna',
)

print(f'\nOptuna Best score  : {lr_optuna_score:.4f}')
print(f'Optuna Best params : {lr_optuna_params}')

---

## 9. Optuna — Random Forest

In [ ]:
def rf_optuna_objective(trial):
    """Optuna objective for Random Forest hyperparameter optimisation.

    Args:
        trial: optuna.Trial object for parameter suggestion.

    Returns:
        Mean cross-validated macro F1 score.
    """
    n_estimators    = trial.suggest_int('n_estimators',    50,  500, step=50)
    max_depth       = trial.suggest_int('max_depth',        5,   30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf',  1,  5)
    max_features    = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    model = SKLearnRF(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )
    scores = cross_val_score(
        model, X_tune_em, y_tune_em,
        cv=TUNE_CFG['cv_folds'],
        scoring=TUNE_CFG['scoring'],
        n_jobs=1,
    )
    return float(np.mean(scores))

print_section_header(f'Optuna — Random Forest (n_trials={TUNE_CFG["n_trials"]})')
rf_optuna_params, rf_optuna_score = tuner.tune_optuna(
    objective_fn=rf_optuna_objective,
    model_name='random_forest_optuna',
)

print(f'\nOptuna Best score  : {rf_optuna_score:.4f}')
print(f'Optuna Best params : {rf_optuna_params}')

---

## 10. Optuna — XGBoost

In [ ]:
def xgb_optuna_objective(trial):
    """Optuna objective for XGBoost hyperparameter optimisation.

    Args:
        trial: optuna.Trial object for parameter suggestion.

    Returns:
        Mean cross-validated macro F1 score.
    """
    n_estimators      = trial.suggest_int('n_estimators',       100,  600, step=50)
    learning_rate     = trial.suggest_float('learning_rate',    0.01,  0.3, log=True)
    max_depth         = trial.suggest_int('max_depth',            3,    9)
    subsample         = trial.suggest_float('subsample',         0.5,  1.0)
    colsample_bytree  = trial.suggest_float('colsample_bytree',  0.5,  1.0)
    reg_alpha         = trial.suggest_float('reg_alpha',         0.0,  2.0)
    reg_lambda        = trial.suggest_float('reg_lambda',        0.5,  3.0)

    model = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    scores = cross_val_score(
        model, X_tune_em, y_tune_em,
        cv=TUNE_CFG['cv_folds'],
        scoring=TUNE_CFG['scoring'],
        n_jobs=1,
    )
    return float(np.mean(scores))

print_section_header(f'Optuna — XGBoost (n_trials={TUNE_CFG["n_trials"]})')
xgb_optuna_params, xgb_optuna_score = tuner.tune_optuna(
    objective_fn=xgb_optuna_objective,
    model_name='xgboost_optuna',
)

print(f'\nOptuna Best score  : {xgb_optuna_score:.4f}')
print(f'Optuna Best params : {xgb_optuna_params}')

---

## 11. Performance Comparison

In [ ]:
# ── Tuning strategy comparison table ──────────────────────────────────────────
comparison_df = tuner.compare_results()
print('Tuning Results (sorted by best_score descending):')
comparison_df

In [ ]:
# ── Visualise: strategy comparison by model ────────────────────────────────────
fig = px.bar(
    comparison_df,
    x='model',
    y='best_score',
    color='method',
    barmode='group',
    title='Hyperparameter Tuning — Best CV Score by Model and Strategy',
    template='plotly_dark',
    labels={'best_score': f'Best {TUNE_CFG["scoring"]}', 'model': 'Model'},
)
fig.update_yaxes(range=[0.0, 1.0])
fig.show()

In [ ]:
# ── Tuning time comparison ─────────────────────────────────────────────────────
fig2 = px.bar(
    comparison_df,
    x='model',
    y='time',
    color='method',
    barmode='group',
    title='Hyperparameter Tuning — Wall-Clock Time by Strategy (seconds)',
    template='plotly_dark',
    labels={'time': 'Time (s)', 'model': 'Model'},
)
fig2.show()

# Save to CSV
tuner.save_results(DIR_OUTPUTS / 'tuning_results.csv')
print('✅ Tuning results saved.')

---

## 12. Best Parameters Summary

In [ ]:
# ── Compile best parameters per model into a JSON artefact ────────────────────
best_params_summary = {
    'logistic_regression': {
        'grid_search':        lr_grid_params,
        'grid_best_score':    lr_grid_score,
        'optuna':             lr_optuna_params,
        'optuna_best_score':  lr_optuna_score,
    },
    'linear_svm': {
        'grid_search':        svm_grid_params,
        'grid_best_score':    svm_grid_score,
    },
    'random_forest': {
        'random_search':       rf_rand_params,
        'random_best_score':   rf_rand_score,
        'optuna':              rf_optuna_params,
        'optuna_best_score':   rf_optuna_score,
    },
    'xgboost': {
        'random_search':       xgb_rand_params,
        'random_best_score':   xgb_rand_score,
        'optuna':              xgb_optuna_params,
        'optuna_best_score':   xgb_optuna_score,
    },
}

save_json(best_params_summary, DIR_OUTPUTS / 'best_hyperparameters.json')
print('✅ Best hyperparameters saved → outputs/best_hyperparameters.json')

# Display as formatted table
rows = []
for model, info in best_params_summary.items():
    for method, val in info.items():
        if 'score' in method:
            rows.append({'Model': model, 'Strategy': method.replace('_best_score',''), 'Best Score': val})
pd.DataFrame(rows).pivot(index='Model', columns='Strategy', values='Best Score')

---

## 13. Save Tuned Models

In [ ]:
# ── Re-train each model with the best Optuna parameters on the full tuning set ─
# Logistic Regression — best from Optuna
lr_tuned = LogisticRegression(
    **lr_optuna_params,
    max_iter=2000,
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
lr_tuned.fit(X_tune_tf, y_tune_tf)
lr_tuned_path = DIR_TUNED_MODELS / 'logistic_regression_tuned.joblib'
joblib.dump(lr_tuned, lr_tuned_path)
print(f'✅ LR tuned model saved → {lr_tuned_path.name}')

In [ ]:
# Linear SVM — best from GridSearch
svm_tuned = svm_best_estimator   # Already fitted by GridSearchCV
svm_tuned_path = DIR_TUNED_MODELS / 'linear_svm_tuned.joblib'
joblib.dump(svm_tuned, svm_tuned_path)
print(f'✅ SVM tuned model saved → {svm_tuned_path.name}')

In [ ]:
# Random Forest — best from Optuna
rf_tuned = SKLearnRF(
    **rf_optuna_params,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
rf_tuned.fit(X_tune_em, y_tune_em)
rf_tuned_path = DIR_TUNED_MODELS / 'random_forest_tuned.joblib'
joblib.dump(rf_tuned, rf_tuned_path)
print(f'✅ RF tuned model saved → {rf_tuned_path.name}')

In [ ]:
# XGBoost — best from Optuna
xgb_tuned = XGBClassifier(
    **xgb_optuna_params,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
xgb_tuned.fit(X_tune_em, y_tune_em)
xgb_tuned_path = DIR_TUNED_MODELS / 'xgboost_tuned.joblib'
joblib.dump(xgb_tuned, xgb_tuned_path)
print(f'✅ XGB tuned model saved → {xgb_tuned_path.name}')

print('\n✅ All tuned models saved.')

---

## 14. Notebook Summary

### Tuning Strategy Comparison

| Strategy | Pros | Cons | Best Use Case |
|---|---|---|---|
| **GridSearchCV** | Exhaustive, reproducible | Exponential time complexity | Small parameter spaces (LR, SVM) |
| **RandomizedSearchCV** | Fast, scales well | No convergence guarantee | Large spaces (RF, XGB) |
| **Optuna** | Efficient, adaptive, visualisable | Requires optuna install | All models — best efficiency/quality |

### Key Findings (to be completed after execution)
- LR best C: **TBD** (Grid vs Optuna comparison)
- RF best n_estimators: **TBD**
- XGBoost improvement over default: **TBD**
- Recommended strategy for this dataset: **Optuna** (best score/time ratio)

### Saved Artefacts
| File | Description |
|---|---|
| `outputs/tuning_results.csv` | All tuning run scores and times |
| `outputs/best_hyperparameters.json` | Best params per model |
| `models/tuned/*.joblib` | Tuned, fitted model files |

→ **Notebook 08**: Model Comparison — compare all trained variants side by side

---
*Fingerprint Project — Hyperparameter Tuning — Complete*